# Survey Stimulus Generation — Attestation Trust Study

**AI assistance disclosure:** This stimulus-generation tooling was built with the assistance of an AI assistant (Claude) for the code scaffolding (parsing, validation, file output) and tutoring how to use the API and prompt correctly. The generation prompt, the attestation-display wording, and all curation decisions are the author's own (Harry Staley). Per the study's documented method, an LLM generates candidate stimuli which the author then curates. Use of generative AI follows the CS 6795 course policy.

In [281]:
from __future__ import annotations

import json
import re
import time
from collections import Counter
from typing import List, Tuple, TypedDict
from pathlib import Path

import pandas as pd
from openai import OpenAI
from IPython.display import display
from dotenv import load_dotenv
from datetime import datetime
load_dotenv()
if not load_dotenv():
    print("WARNING: .env not found; run setup_env.py to create it.")
print(f"Key loaded: {load_dotenv()}")
print("NOTE: Be sure that you have a .env file with your OpenAI API key.")

Key loaded: True
NOTE: Be sure that you have a .env file with your OpenAI API key.


In [282]:
MODEL: str = "gpt-5.5"            # model name
# NOTE: Temperature is not available in gpt-5.5, but it is in others.
# TEMPERATURE: float = 0.7        # controls randomness; higher = more varied output
MAX_RETRIES: int = 5            # how many times to retry on hard failure
LENGTH_RATIO_WARN: float = 0.25 # warn if answers differ >25% in length
SENTENCE_DIFF_WARN: int = 1     # warn if sentence counts differ by >1

In [283]:
class Stem(TypedDict):
    """Schema for one generated survey stimulus."""
    stem_id: int
    stakes: str
    topic: str
    question_text: str
    correct_answer: str
    incorrect_answer: str
    source_name: str
    source_citation: str
    source_url: str
    ground_truth_note: str

# Verifies that the JSON object has the required keys as defined in the Stem class.
REQUIRED_KEYS: set[str] = {
    "stem_id", "stakes", "topic", "question_text",
    "correct_answer", "incorrect_answer", "source_name",
    "source_citation", "source_url", "ground_truth_note",
}

In [284]:
prompt_template = Path("generation_prompt.md").read_text(encoding="utf-8")
GENERATION_PROMPT = prompt_template.format(
    schema_fields=", ".join(Stem.__annotations__)
)

In [285]:
def attestation_text(att_level: str, item: Stem) -> str:
    """Return the rendered attestation display for a given attestation level."""
    if att_level == "none":
        return ""
    if att_level == "weak":
        return f"Source: {item['source_name']} — {item['source_citation']}"
    return (f"Source: {item['source_name']} — {item['source_citation']}\n"
            f"Publisher verified ({item['source_name']})\n"
            f"Document unaltered since publication\n"
            f"Independently checked for relevance")

In [286]:
def parse_json(raw_text: str) -> List[Stem]:
    """Parse model output into stems; recover the [...] array if wrapped."""
    raw_text = raw_text.strip()
    raw_text = re.sub(r"^```(?:json)?|```$", "", raw_text, flags=re.MULTILINE).strip()
    try:
        return json.loads(raw_text)
    except json.JSONDecodeError:
        start, end = raw_text.find("["), raw_text.rfind("]") + 1
        if start == -1 or end == 0:
            raise
        return json.loads(raw_text[start:end])

In [287]:
def sentence_count(text: str) -> int:
    """Rough sentence count, robust to decimals/abbreviations (for warnings only)."""
    if not text.strip():
        return 0
    t = re.sub(r"\d+\.\d+", "0", text)
    for abbr in ("Dr.", "Mr.", "Mrs.", "Ms.", "U.S.", "U.K.", "e.g.", "i.e.",
                 "etc.", "mg.", "mL.", "vs.", "Inc.", "Ltd.", "Fig.", "No."):
        t = t.replace(abbr, abbr.replace(".", ""))
    return max(len(re.findall(r"[.!?]+", t)), 1)

In [288]:
def validate_stems(items: List[Stem]) -> Tuple[List[str], List[str]]:
    """Return (errors, warnings). Errors trigger retry; warnings flag for curation."""
    errors: List[str] = []
    warnings: List[str] = []

    if len(items) != 12:
        errors.append(f"Expected 12 stems, found {len(items)}.")
    low = sum(x.get("stakes") == "low" for x in items)
    high = sum(x.get("stakes") == "high" for x in items)
    if low != 6 or high != 6:
        errors.append(f"Expected 6 low / 6 high; found {low} low / {high} high.")
    if sorted(x.get("stem_id", -1) for x in items) != list(range(1, 13)):
        errors.append("stem_id values must be 1..12 with no gaps/dupes.")

    topics: List[str] = []
    for item in items:
        sid = item.get("stem_id", "?")
        if set(item.keys()) != REQUIRED_KEYS:
            errors.append(f"Stem {sid} schema mismatch.")
            continue
        topics.append(item["topic"].lower())
        ca, ia = item["correct_answer"], item["incorrect_answer"]
        if ia.strip() == "REFUSED_NEEDS_MANUAL" or not ia.strip():
            errors.append(f"Stem {sid} incorrect_answer REFUSED -- build manually.")
            continue
        if abs(sentence_count(ca) - sentence_count(ia)) > SENTENCE_DIFF_WARN:
            warnings.append(f"Stem {sid}: sentence-count mismatch (review).")
        if max(len(ca), len(ia)) and abs(len(ca) - len(ia)) / max(len(ca), len(ia)) > LENGTH_RATIO_WARN:
            warnings.append(f"Stem {sid}: answer-length mismatch (review).")

    dups = [t for t, c in Counter(topics).items() if c > 1]
    if dups:
        errors.append(f"Duplicate topics: {dups}")
    return errors, warnings

In [289]:
# Test the whole logic chain with fake data -- no API, no cost.
_mock = [
    {"stem_id": i, "stakes": "low" if i <= 6 else "high", "topic": f"topic{i}",
     "question_text": "Q?", "correct_answer": "A true statement here.",
     "incorrect_answer": "A false statement here.", "source_name": "Src",
     "source_citation": "Doc, src.org", "source_url": "https://src.org",
     "ground_truth_note": "note"}
    for i in range(1, 13)
]
_errors, _warnings = validate_stems(_mock)
print("errors:", _errors)
print("warnings:", _warnings)
assert not _errors, "mock should pass structural validation"
print("MOCK PASSED — logic chain works.")

errors: []
warnings: []
MOCK PASSED — logic chain works.


In [290]:
def generate_response() -> List[Stem]:
    """One generation call; stem_ids are assigned in code, not trusted from the model."""
    response = client.responses.create(
        model=MODEL,
        input=GENERATION_PROMPT,
    )
    items = parse_json(response.output_text)
    # assign stem_ids by position -- the model is unreliable at sequential numbering
    for i, item in enumerate(items, start=1):
        item["stem_id"] = i
    return items

In [291]:
_test = client.responses.create(
    model=MODEL,
    input='Return exactly this JSON and nothing else: [{"ok": 1}]',
)
print(repr(_test.output_text))

'[{"ok": 1}]'


In [292]:
def generate_with_retries() -> Tuple[List[Stem], List[str]]:
    """Generate stems; retry on HARD failures, surface warnings for curation."""
    last_errors: List[str] = []
    for attempt in range(1, MAX_RETRIES + 1):
        print("=" * 80)
        print(f"ATTEMPT {attempt}/{MAX_RETRIES}")
        try:
            items = generate_response()
            errors, warnings = validate_stems(items)
            if not errors:
                print("Structural validation passed.")
                if warnings:
                    print(f"\n{len(warnings)} item(s) flagged for human curation:")
                    for w in warnings:
                        print("  ~", w)
                else:
                    print("No curation warnings.")
                print()
                return items, warnings
            print("Hard failures (regenerating):")
            for e in errors:
                print("  -", e)
            last_errors = errors
        except Exception as e:
            print("Generation failed:", str(e))
            last_errors = [str(e)]
        time.sleep(1)
    raise RuntimeError(
        "Generation failed after retries. Last hard failures:\n"
        + "\n".join(last_errors)
        + "\n\nIf failures are REFUSED incorrect answers on sensitive items, "
          "generate the rest and CONSTRUCT those items manually (document it)."
    )


In [293]:
def build_loop_table(items: List[Stem]) -> pd.DataFrame:
    """Expand validated stems into the 72-row Qualtrics loop table."""
    rows: List[dict] = []
    for item in items:
        for att_level in ("none", "weak", "strong"):
            for correctness in ("correct", "incorrect"):
                answer = (item["correct_answer"] if correctness == "correct"
                          else item["incorrect_answer"])
                rows.append({
                    "stem_id": item["stem_id"],
                    "stakes": item["stakes"],
                    "question_text": item["question_text"],
                    "answer_text": answer,
                    "att_level": att_level,
                    "correctness": correctness,
                    "attestation_text": attestation_text(att_level, item),
                })
    return pd.DataFrame(rows)

In [294]:
items, warnings = generate_with_retries()

stems_df = pd.DataFrame(items)
loop_df = build_loop_table(items)

# show results
print("\n" + "=" * 80 + "\nGENERATED STEMS\n" + "=" * 80)
print(stems_df.to_string(index=False))
print("\n" + "=" * 80 + "\nQUALTRICS LOOP TABLE\n" + "=" * 80)
print(loop_df.to_string(index=False))

display(stems_df)
display(loop_df)

# write outputs to a dedicated directory
out = Path("output")
out.mkdir(exist_ok=True)
ts = datetime.now().strftime("%Y-%m-%dT%H%M%S")

stems_df.to_csv(out / f"generated_stems{ts}.csv", index=False)
loop_df.to_csv(out / f"qualtrics_loop_table{ts}.csv", index=False)
with open(out / f"generated_stems{ts}.json", "w", encoding="utf-8") as f:
    json.dump(items, f, indent=2)

metadata = {
    "model": MODEL,
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "max_retries": MAX_RETRIES,
    "curation_warnings": warnings,
    "note": "Soft warnings indicate items flagged for human curation "
            "(matched length/sentence count), per the documented method.",
}
with open(out / f"generation_metadata{ts}.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(f"\nSaved to {out}/: generated_stems{ts}.json, generated_stems{ts}.csv, "
      f"qualtrics_loop_table{ts}.csv, generation_metadata{ts}.json")
if warnings:
    print(f"\n{len(warnings)} item(s) need curation review (see {out}/generation_metadata{ts}.json).")

ATTEMPT 1/5
Structural validation passed.
No curation warnings.


GENERATED STEMS
 stem_id stakes                            topic                                                                                                                                                                                  question_text                                                                                      correct_answer                                                                              incorrect_answer                                    source_name                                                               source_citation                                                                                          source_url                                                                                                                                           ground_truth_note
       1    low            measurement standards                                                       

,stem_id,stakes,topic,question_text,correct_answer,incorrect_answer,source_name,source_citation,source_url,ground_truth_note
0,1,low,measurement standards,"In the International System of Units, how is t...",The meter is defined by the distance light tra...,The meter is defined by the distance light tra...,National Institute of Standards and Technology,"NIST, SI Units: Length, meter definition.",https://www.nist.gov/pml/owm/metric-si/si-units,NIST states that the meter is defined using th...
1,2,low,human physiology,"In normal human circulation, which heart chamb...",The right ventricle pumps blood from the heart...,The left ventricle pumps blood from the heart ...,"National Heart, Lung, and Blood Institute","NHLBI, How the Heart Works.",https://www.nhlbi.nih.gov/health/heart/how-hea...,NHLBI explains that the right ventricle pumps ...
2,3,low,volcanic rocks,Which common volcanic rock can often float bec...,Pumice can float because its many gas bubbles ...,Scoria can float because its many gas bubbles ...,U.S. Geological Survey,"USGS, common volcanic rocks FAQ.",https://www.usgs.gov/faqs/what-are-some-common...,"USGS describes pumice as a very light, porous ..."
3,4,low,constitutional history,When was the Twenty-Seventh Amendment to the U...,The Twenty-Seventh Amendment was ratified in 1...,The Twenty-Seventh Amendment was ratified in 1...,National Archives,"National Archives, 27th Amendment milestone do...",https://www.archives.gov/milestone-documents/2...,The National Archives identifies 1992 as the y...
4,5,low,American music history,When did Congress make “The Star-Spangled Bann...,Congress made “The Star-Spangled Banner” the n...,Congress made “The Star-Spangled Banner” the n...,Library of Congress,"Library of Congress, The Star-Spangled Banner.",https://www.loc.gov/item/ihas.200000017/,The Library of Congress notes that Congress de...
5,6,low,English literature,How was Mary Shelley’s Frankenstein first publ...,Frankenstein was first published anonymously i...,Frankenstein was first published under Mary Sh...,The British Library,"British Library, the first edition of Frankens...",https://www.bl.uk/romantics-and-victorians/art...,The British Library describes the 1818 first e...
6,7,high,retirement account tax penalties,For a taxpayer under age 59½ who takes a non-e...,The distribution is generally subject to a 10%...,The distribution is generally subject to a 5% ...,Internal Revenue Service,"IRS Publication 590-B, Distributions from Indi...",https://www.irs.gov/publications/p590b,IRS Publication 590-B states that early IRA di...
7,8,high,COBRA health coverage election,"After receiving a COBRA election notice, how l...",A qualified beneficiary generally has 60 days ...,A qualified beneficiary generally has 30 days ...,U.S. Department of Labor,"DOL, COBRA Continuation Coverage.",https://www.dol.gov/general/topic/health-plans...,The Department of Labor states that qualified ...
8,9,high,Federal Tort Claims Act deadline,"Under the Federal Tort Claims Act, by when mus...",The claim must be presented to the agency with...,The claim must be presented to the agency with...,U.S. Code,28 U.S.C. § 2401(b).,https://uscode.house.gov/view.xhtml?req=granul...,28 U.S.C. § 2401(b) provides that a tort claim...
9,10,high,Selective Service registration,"Under federal Selective Service rules, when mu...",Most male U.S. citizens and male immigrants mu...,Most male U.S. citizens and male immigrants mu...,Selective Service System,"Selective Service System, Who Needs to Register.",https://www.sss.gov/register/who-needs-to-regi...,The Selective Service System states that most ...


,stem_id,stakes,question_text,answer_text,att_level,correctness,attestation_text
0,1,low,"In the International System of Units, how is t...",The meter is defined by the distance light tra...,none,correct,
1,1,low,"In the International System of Units, how is t...",The meter is defined by the distance light tra...,none,incorrect,
2,1,low,"In the International System of Units, how is t...",The meter is defined by the distance light tra...,weak,correct,Source: National Institute of Standards and Te...
3,1,low,"In the International System of Units, how is t...",The meter is defined by the distance light tra...,weak,incorrect,Source: National Institute of Standards and Te...
4,1,low,"In the International System of Units, how is t...",The meter is defined by the distance light tra...,strong,correct,Source: National Institute of Standards and Te...
...,...,...,...,...,...,...,...
67,12,high,"Under federal consumer rules, if a lost debit ...",The consumer’s usual maximum liability is $500.,none,incorrect,
68,12,high,"Under federal consumer rules, if a lost debit ...",The consumer’s usual maximum liability is $50.,weak,correct,"Source: Federal Trade Commission — FTC, Lost o..."
69,12,high,"Under federal consumer rules, if a lost debit ...",The consumer’s usual maximum liability is $500.,weak,incorrect,"Source: Federal Trade Commission — FTC, Lost o..."
70,12,high,"Under federal consumer rules, if a lost debit ...",The consumer’s usual maximum liability is $50.,strong,correct,"Source: Federal Trade Commission — FTC, Lost o..."



Saved to output/: generated_stems.json, generated_stems.csv, qualtrics_loop_table.csv, generation_metadata.json
